In [3]:
import triton 
import torch
import triton.language as tl

In [2]:
@triton.jit
def kernel(
        x_ptr,
        y_ptr,
        out_ptr,
        n_eles,
        BLOCK_DIM: tl.constexpr
):
    pid = tl.program_id(0)
    # Offsets for this program instance
    offsets = pid * BLOCK_DIM + tl.arange(0, BLOCK_DIM) 
    mask = offsets < n_eles

    # Load data
    a = tl.load(x_ptr + offsets, mask=mask)
    b = tl.load(y_ptr + offsets, mask=mask)

    # Calculate c = a + 2^b
    c = a + tl.exp2(b)

    # Store result
    tl.store(out_ptr + offsets, c, mask=mask)

# 1. Setup Data
x = torch.rand((32, 1), device='cuda')
y = torch.rand((32, 1), device='cuda')
n_eles = x.numel()
out = torch.empty((32, 1), device='cuda')
BLOCK_DIM = 1024

# 2. Define Grid (3 blocks on axis 0, 2 on axis 1)
grid = (3, 2, 1)

# 3. Launch Kernel
kernel[grid](x, y, out, n_eles, BLOCK_DIM)

# 4. Return/Print Output
# Move from GPU to CPU to see the values
print("Input X sample:", x[0].item())
print("Input Y sample:", y[0].item())
print("Result C (first 5 elements):")
print(out[:5].cpu())

# Verification check
expected = x + torch.pow(2, y)
torch.testing.assert_close(out, expected)
print("\nSuccess: Triton output matches PyTorch expected output!")

Input X sample: 0.8550713658332825
Input Y sample: 0.006093887612223625
Result C (first 5 elements):
tensor([[1.8593],
        [2.0908],
        [2.6448],
        [1.9308],
        [2.5725]])

Success: Triton output matches PyTorch expected output!


In [23]:
@triton.jit
def add_stride_kernel(
    x_ptr, y_ptr, out_ptr,
    n_elements,
    stride, 
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    
    # We point to the position [0, stride, 2*stride] instead of [0,1,2] with offset only
    strided_offset = offsets * stride
    
    mask = offsets < n_elements # Offsets is an indicies for our memory, so it must be lower than n_ele rather than strided offset which stride in memory
    
    x = tl.load(x_ptr + strided_offset, mask=mask)
    y = tl.load(y_ptr + strided_offset, mask=mask)
    
    output = x + y
    
    tl.store(out_ptr + strided_offset, output, mask=mask)

def add(x, y, stride):
    out = torch.empty_like(x)

    n_elements = x.numel() // stride # we use quotient to find the number of elements to process, due to striding we have less elements to process

    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )

    add_stride_kernel[grid](x, y, out, n_elements, stride, BLOCK_SIZE=1024)

    return out

stride_val = 2
x = torch.tensor([1, 2, 2, 0, 3, 0, 4, 0, 5, 0], device='cuda', dtype=torch.float32)
y = torch.tensor([10, 4, 20, 0, 30, 0, 40, 0, 50, 0], device='cuda', dtype=torch.float32)

result_kernel = add(x, y, stride_val)
result_torch = x + y # 

print(f"result_torch: {result_torch}")
print(f"Kernel Result: {result_kernel}")

result_torch: tensor([11.,  6., 22.,  0., 33.,  0., 44.,  0., 55.,  0.], device='cuda:0')
Kernel Result: tensor([11.,  0., 22.,  0., 33.,  0., 44.,  0., 55.,  0.], device='cuda:0')


In [5]:
torch.cuda.empty_cache()

In [6]:
@triton.jit
def matmul_kernel(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # Strides
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
):
    # 1. Launch 2d block 
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # 2. Create offset vectors
    offs_m = tl.arange(0, BLOCK_SIZE_M)
    offs_n = tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    # 3. Create the grid of indices for this block
    curr_m = (pid_m * BLOCK_SIZE_M) + offs_m # this will be matrix offset for M at dim 0
    curr_n = (pid_n * BLOCK_SIZE_N) + offs_n # this will be matrix offset for N at dim 1

    acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    # 4. Iterate through K to calculate dot product for each tile
    for k in range(0, K, BLOCK_SIZE_K):
        # Update current K indices for this iteration
        curr_k = k + offs_k
        
        # Create 2D pointer blocks by broadcasting M and K indices
        # Shape of a_ptrs: (BLOCK_SIZE_M, BLOCK_SIZE_K)
        a_ptrs = a_ptr + (curr_m[:, None] * stride_am + curr_k[None, :] * stride_ak) 
        
        # Shape of b_ptrs: (BLOCK_SIZE_K, BLOCK_SIZE_N)
        b_ptrs = b_ptr + (curr_k[:, None] * stride_bk + curr_n[None, :] * stride_bn)

        # Load blocks with masking to handle cases where M, N, or K aren't multiples of block sizes
        a_block = tl.load(a_ptrs, mask=(curr_m[:, None] < M) & (curr_k[None, :] < K))
        b_block = tl.load(b_ptrs, mask=(curr_k[:, None] < K) & (curr_n[None, :] < N))

        # Matrix multiplication on the blocks
        acc += tl.dot(a_block, b_block)

    # 6. Store the final result back to Global Memory
    # The pointer grid must match the shape of 'acc' (BLOCK_SIZE_M, BLOCK_SIZE_N)
    c_ptrs = c_ptr + (curr_m[:, None] * stride_cm + curr_n[None, :] * stride_cn)
    c_mask = (curr_m[:, None] < M) & (curr_n[None, :] < N)
    
    tl.store(c_ptrs, acc, mask=c_mask)

In [ ]:
torch.set_float32_matmul_precision('highest') 
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False

def matmul(a, b):
    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = lambda META: (
        triton.cdiv(M, META['BLOCK_SIZE_M']), 
        triton.cdiv(N, META['BLOCK_SIZE_N'])
    )
    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=64, BLOCK_SIZE_N=64, BLOCK_SIZE_K=32,
    )
    return c

cuda_generator = torch.Generator(device='cuda').manual_seed(42)
a_mat = torch.randn((128, 64), device='cuda', dtype=torch.float32, generator=cuda_generator)
b_mat = torch.randn((64, 128), device='cuda', dtype=torch.float32, generator=cuda_generator)

c_kernel = matmul(a_mat, b_mat)
c_torch = torch.matmul(a_mat, b_mat)

print("Triton Matmul Result:\n", c_kernel)
print("Torch Matmul", c_torch)
torch.testing.assert_close(c_kernel, c_torch)


/home/dohuyduc/miniconda3/envs/triton/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Triton Matmul Result:
 tensor([[  6.3024,   5.0832,  -1.0065,  ...,   0.0000,   0.0000,   0.0000],
        [ 17.1340,   2.1086,  -3.0265,  ...,   0.0000,   0.0000,   0.0000],
        [  3.9629,   1.5338,   3.1992,  ...,   0.0000,   0.0000,   0.0000],
        ...,
        [  3.6511,  -7.2393,  -4.1667,  ...,   0.0000,   0.0000,   0.0000],
        [  2.7727,  -3.5086,  -7.7233,  ...,   0.0000,   0.0000,   0.0000],
        [ -2.6941,  -6.1312, -13.8773,  ...,   0.0000,   0.0000,   0.0000]],
       device='cuda:0')
Torch Matmul tensor([[  6.3103,   5.0860,  -1.0062,  ...,  -6.1185,   3.5654,  -8.4771],
        [ 17.1457,   2.1097,  -3.0313,  ...,   4.5661,   7.0699,   5.3764],
        [  3.9659,   1.5323,   3.2038,  ...,   1.9161,  -5.9991,  -5.7718],
        ...,
        [  3.6513,  -7.2427,  -4.1708,  ...,   7.9497,  -3.0065,  -0.1670],
        [  2.7735,  -3.5068,  -7.7250,  ...,  -2.5290,  -7.4924, -13.9862],
        [ -2.6971,  -6.1373, -13.8907,  ...,  -2.7239,  13.7436,   0.7782]],


AssertionError: Tensor-likes are not close!

Mismatched elements: 16371 / 16384 (99.9%)
Greatest absolute difference: 32.96298599243164 at index (37, 87) (up to 1e-05 allowed)
Greatest relative difference: 6.957409381866455 at index (7, 6) (up to 1.3e-06 allowed)

In [10]:
@triton.jit
def matmul_kernel(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        stride_am, stride_ak,  #
        stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, accumulator, mask=c_mask)

In [ ]:
@triton.jit
def matmul_kernel(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # Strides
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
):
    # 1. Launch 2d block 
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # 2. Create offset for each vector
    offs_m = (pid_m * BLOCK_SIZE_M) + tl.arange(0, BLOCK_SIZE_M) # this will be matrix offset for M at dim 0
    offs_n = (pid_n * BLOCK_SIZE_N) + tl.arange(0, BLOCK_SIZE_N) # this will be matrix offset for N at dim 1
    offs_k = tl.arange(0, BLOCK_SIZE_K) # we dont use pid for the output

    acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    # 4. Iterate through K to calculate dot product for each tile
    for k in range(0, K, BLOCK_SIZE_K): # over K dimension with step is BLOCK_SIZE_K to jump to next tile
        # Update current K indices for this iteration
        curr_k = k + offs_k
        
        # Create 2D pointer blocks by broadcasting M and K indices
        # as a tile, the a pointers will point to the 1st dim of M offset and 2nd dim of K offset because the M denote the address of row and K denote the address of column in A matrix
        # The multiply by stride for correct memory offset
        # This implementation only the k move dynamically where the k is incremented in each loop
        # Shape of a_ptrs: (BLOCK_SIZE_M, BLOCK_SIZE_K)
        a_ptrs = a_ptr + (offs_m[:, None] * stride_am + curr_k[None, :] * stride_ak) 
        
        # Shape of b_ptrs: (BLOCK_SIZE_K, BLOCK_SIZE_N)
        b_ptrs = b_ptr + (curr_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

        # Load blocks with masking to handle cases where M, N, or K aren't multiples of block sizes
        a_block = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (curr_k[None, :] < K))
        b_block = tl.load(b_ptrs, mask=(curr_k[:, None] < K) & (offs_n[None, :] < N))

        # Matrix multiplication on the blocks
        acc += tl.dot(a_block, b_block)

    # 6. Store the final result back to Global Memory
    # The pointer grid must match the shape of 'acc' (BLOCK_SIZE_M, BLOCK_SIZE_N)
    c_ptrs = c_ptr + (offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn)
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    
    tl.store(c_ptrs, acc, mask=c_mask)

In [ ]:

@triton.jit
def matmul_kernel(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # -----------------------------------------------------------
    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # You can fuse arbitrary activation functions here
    # while the accumulator is still in FP32!

    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

In [9]:
@torch.compile(options={"triton.cudagraphs": True}, fullgraph=True)
def softmax(x,y):
    x = x + y
    return torch.nn.functional.softmax(x, dim=-1)

x = torch.randn((1024, 512), device='cuda', dtype=torch.float32)
y = torch.randn((1024, 512), device='cuda', dtype=torch.float32)
out = softmax(x, y)
print("Softmax Output Sample:", out[0, :5].cpu())

Softmax Output Sample: tensor([0.0034, 0.0007, 0.0002, 0.0031, 0.0002])


In [5]:
import helion
from helion._testing import DEVICE
from helion._testing import run_example
import helion.language as hl

@helion.kernel()
def add_kernel(
    x : torch.Tensor,
    y : torch.Tensor,
):
    x = hl.load(x) 
    y = hl.load(y)
    
    out = torch.empty_like(x)
    out = x + y
    hl.store(out)
    

x = torch.randn([1024, 1024], device=DEVICE, dtype=torch.float16)
y = torch.randn([1024, 1024], device=DEVICE, dtype=torch.float16)
out = run_example(add_kernel, x, y)
print(out)

TypeError: 'Tensor' object is not callable